In [6]:
import os
import getpass
from dotenv import load_dotenv
load_dotenv(override=True)


if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

# 🤖 Agents in LangChain

**LangChain** is a Python library for building apps on top of language models (LLMs). **LangGraph** is the lower-level library underneath it. It runs the steps of an app and manages its data. You will see both names in this notebook.

**Agents** are systems where a language model decides what to do next, such as calling a search tool, and then uses the result to answer. They suit tasks like Q&A and interacting with other services.

### How an agent works

1. The model reads the question and decides whether it needs a tool.
2. If it does, it calls the tool (for example, a web search).
3. It reads the tool's result.
4. It repeats steps 1 to 3 if needed, then gives a final answer.

### 🆚 Agents vs. Chains

- **Agents** dynamically decide on actions using language models.
- **Chains** have a predetermined sequence of actions.

Agents are about smart decision-making and interacting with the digital environment, while chains are about a pre-set flow of information.

### 🛠️ Tools and Toolkits

- **Tools** are like an agent's skills, specific to tasks like data retrieval.
- **Toolkits** bundle these skills for more complex tasks.

### 🧠 The Role of Memory

- By default, an agent forgets everything between separate calls. Memory lets it keep past interactions across calls. We cover this at the end of the notebook.



### Setting up the model

- The first code cell loads your [OpenRouter](https://openrouter.ai) API key, from a `.env` file or by asking you for it. OpenRouter gives you one API key for many language models.
- `ChatOpenRouter` is the LangChain wrapper that sends requests to OpenRouter.
- `temperature` controls how random the output is. `0.0` gives the most consistent answers, and higher values give more varied ones.
- The model name comes from the `MODEL` environment variable, or falls back to a default. You can find model names on [openrouter.ai/models](https://openrouter.ai/models).


In [40]:
from langchain_openrouter import ChatOpenRouter

openrouter_model = os.getenv("MODEL", "openai/gpt-6-luna")  # swap on openrouter.ai/models if needed
# instantiate the llm
llm = ChatOpenRouter(model=openrouter_model, temperature=0.0)

# Giving the Agent Tools


 - 🛠️ 🔧 Tools are integrated into agents either all at once during setup or individually as needed.

 - 🧰 Opt for LangChain's ready-made tools or craft your own custom tools to meet your unique needs.

A **tool** is a function the agent is allowed to call. The model sees each tool's name and description, and decides when to use it.

The two tools in the next code cell are DuckDuckGo search and Wikipedia. They make live calls to the internet, so results can change over time and may be incomplete or wrong. Check important answers against a source.

You can find the available native tools [here](https://docs.langchain.com/oss/python/integrations/tools).

In [42]:
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper, WikipediaAPIWrapper
import wikipedia as wiki_api

# Wikipedia blocks requests that do not identify themselves with a User-Agent (403 error).
# Replace the contact details with your own.
wiki_api.set_user_agent("CC-AIEng-notebook/1.0 (IADT student project; contact: your-email@example.com)")

# DuckDuckGo search. It asks several search engines and merges their results. We leave out the
# 'wikipedia' engine, which builds its address from the region and can fail (for example, wt.wikipedia.org).
search = DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region="us-en", backend="bing,brave,duckduckgo,google,mojeek,yahoo,yandex"))

# Wikipedia
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=None))

# Wrap each tool so a failed lookup returns a short message instead of crashing the whole agent run.
# The model reads the message and can tell the user what went wrong.
from langchain.tools import tool

@tool
def web_search(query: str) -> str:
    """Search the web with DuckDuckGo. Use it for recent news or general facts."""
    try:
        return search.run(query)
    except Exception as e:
        return f"Web search failed ({type(e).__name__}). Tell the user the search did not work. Do not invent results."

@tool
def wikipedia_search(query: str) -> str:
    """Look up a topic on Wikipedia. Use it for background facts about people, places and ideas."""
    try:
        return wikipedia.run(query)
    except Exception as e:
        return f"Wikipedia lookup failed ({type(e).__name__}). Tell the user the lookup did not work. Do not invent results."

# equip it with some tools
tools = [web_search, wikipedia_search]

# Create the agent

**Setting Up an Agent:**

The model is the reasoning engine of your agent: it reads the conversation and decides what to do next. An agent is built from:

- **Model**: the language model it uses.
- **Tools**: the functions it can call.
- **System prompt**: standing instructions sent to the model on every call, for example "You are a helpful assistant...".

Models can be chosen in two ways:

1. **Static Model** 🛠️: The model is set once when you create the agent and does not change during a run. This is the simplest approach, and the one used in the next example.

2. **Dynamic model** 🚀: The model is chosen at runtime, based on the current state or context. This allows routing logic, for example using a cheaper model for simple questions. We cover it after the next example.


In [43]:
query = """
Who is the current Chief AI Scientist at Meta AI? When was he born? Where was he born? What's the current temperature there?
"""

### Running the agent

The agent receives the question and decides which tools to use. For this question, it may search the web and check Wikipedia before answering.

`print(result)` shows the full raw output, a dictionary that contains the list of messages. Later examples use `pretty_print()` to make the messages easier to read.

The model has no real weather tool, so the answer to "What's the current temperature there?" depends on what the search finds. Check it against a reliable source.


In [45]:
# initialize the agent
from langchain.agents import create_agent


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can search for information using available tools."
)

# Run with messages format
result = agent.invoke({
    "messages": [{"role": "user", "content": query}]
})

print(result)


{'messages': [HumanMessage(content="\nWho is the current Chief AI Scientist at Meta AI? When was he born? Where was he born? What's the current temperature there?\n", additional_kwargs={}, response_metadata={}, id='774bf21b-2a80-4af9-abb7-1d66042e20b6'), AIMessage(content='', additional_kwargs={'reasoning_details': [{'data': 'gAAAAABq4gCWbs1tnCIyod4LtMtF9_1DuwCgpa-LYHjqWxG8_sm6iCmIZmJjPphmrHqC1mpAuV4cw6koC873ndQvul5ubY95b0aVBWrSnjW65Dne9247SC6_dun1MVl7cOTLu_aiwJvRotY86TZGhyt0wqEOr8mzeUa0Y6zbzhipSLyfHDpPv46yNSMtfW_xGIH6Drm_1-S6BYjNSeoFHAx6GQ7dEEsjA8eaVxYnt8YRB8BbPMjZ6vA-zKofGlgwEt8tqGvuOl3Q1vaxNCZWWAQyx2A9Ngb3w6_gg6rW-gQ3AdCNCYFLw9s6q7jOCQyS1STiCAktVCJbZBFRgrgCsThrX-QyDcY76BkMbE4yg421qbxa7tQsjIYnwwBFQupEnb56Zore7WgVBlMxGbkh_1I4mwRK9LO0m7p-dBQ3ylbgxURChDUKunij3RfjaNpJggo_o_wjYNb8MDVhl1ktjf2RbSh-FFI9bOPcFr3Py3kwF-RHnQ_ye1LwrCMSPufBAhDfqWMoO_tez6bCKZTIxiop1BrjT_igkceUJ8-j2LRM9L5uMnYv_2kPHvQKtsXxgIqjTlxwRn83WlG_zP8mtOLbGO_WitiZmcIfvb0BmrMdNxVDjAf57TY-TzlYyqRvwLv510Bz_uCbpYeSFPMRcsyzJmt_WCvb

In [46]:
from IPython.display import Markdown, display

display(Markdown(result["messages"][-1].content))

You may mean **Yann LeCun**, Meta’s longtime Chief AI Scientist. He was born **July 8, 1960**, in **Soisy-sous-Montmorency, France**. He announced in November 2025 that he planned to leave Meta at the end of that year, so the title may no longer be current.

I couldn’t retrieve a reliable live weather reading, so I can’t confirm the current temperature in Soisy-sous-Montmorency.

### Dynamic model

With a dynamic model, the agent can switch models *during* a run. The code below does this with a **middleware** class: a piece of code that runs around each model call.

- `wrap_model_call` runs each time the agent is about to call the model. It counts the messages in the conversation (`request.state["messages"]`).
- If the conversation has more than `message_threshold` messages (here, 2), it uses the advanced model. Otherwise it uses the basic model.
- A tool call adds its request and its result to the conversation. After a search, the count is above 2.

[Dynamic agent model](https://docs.langchain.com/oss/python/langchain/agents#dynamic-model)

In [48]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.messages import AIMessage
from langchain_openrouter import ChatOpenRouter


basic_model = ChatOpenRouter(model="openai/gpt-5-nano")
advanced_model = ChatOpenRouter(model="openai/gpt-6-luna")


class DynamicModelMiddleware(AgentMiddleware):

    def wrap_model_call(self, request, handler):
        if len(request.state["messages"]) > self.message_threshold:
            request = request.override(model=advanced_model)
        else:
            request = request.override(model=basic_model)

        return handler(request)

    def __init__(self, message_threshold: int) -> None:
        self.message_threshold = message_threshold

agent = create_agent(
    model=basic_model,
    tools=tools,
    middleware=[DynamicModelMiddleware(message_threshold=2)]
)


### Watching the switch

The question below needs a web search, so it shows the switch in action:

1. The first model call sees one message, the question, so the basic model runs. It decides to search.
2. The search results are added to the conversation, which now has more than 2 messages.
3. The next model call uses the advanced model to write the answer.

The cell after the run shows which model wrote each reply.


In [49]:
# The basic model decides to search, then the advanced model writes the answer.
result = agent.invoke({
    "messages": [{"role": "user", "content": "What are the latest advancements in AI research?"}]
})

### Reading the messages

A run returns a list of **messages**. Each one has a type:

- **Human**: your question.
- **AI**: the model's reply, or a request to call a tool.
- **Tool**: the result a tool sent back to the model.

The code below prints every message from the last question above with `pretty_print()`. Tool-call messages often have no text, so they can look empty when printed.


In [50]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What are the latest advancements in AI research?
================================== Ai Message ==================================
Tool Calls:
  web_search (call_5w25Qikatuq5JAGXJXNKbYMo)
 Call ID: call_5w25Qikatuq5JAGXJXNKbYMo
  Args:
    query: latest advancements AI research 2025 2026 foundation models multimodal RLHF alignment safety efficiency sparsity training compute data-centric AI 2025 2026
================================= Tool Message =================================
Name: web_search

An educational summary of late 2025 AI research papers. Learn about advances in foundation models like GPT-5, agentic AI, and new neuromorphic hardware. Discover the top 10 AI research papers of 2025. Explore breakthroughs in reasoning models, autonomous agents, and reinforcement learning. With the pace of development in 2025, from reasoning models, multimodal capabilities, to open-source advancements, the demand 

In [51]:
# Which model wrote each AI reply? Replies that only call a tool have no text, so we show the number of tool calls.
for msg in result["messages"]:
    if msg.type == "ai":
        print(msg.response_metadata.get("model_name"), "| tool calls:", len(msg.tool_calls))


openai/gpt-5-nano | tool calls: 1
openai/gpt-6-luna | tool calls: 5
openai/gpt-6-luna | tool calls: 0


### Writing a tool with `@tool`

The `@tool` decorator turns a normal Python function into a tool the agent can call. The model uses three parts of it to decide how to use the tool:

1. **Function name**: `get_weather`.
2. **Docstring**: the text inside the triple quotes. The model reads it to decide when to use the tool.
3. **Type hints**: `location: str` and `units: Literal["celsius", "fahrenheit"]`. These describe the inputs the model should send.

**Note:** this tool is a stub. It always returns a fixed value (22 degrees Celsius or 72 degrees Fahrenheit) and does not check real weather data. Treat it as a stand-in for a real API call.

In [52]:
from typing import Literal
from langchain.tools import tool


@tool
def get_weather(location: str, units: Literal["celsius", "fahrenheit"] = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## Accessing Context

[Read first](https://docs.langchain.com/oss/python/langchain/tools#accessing-context)

#### Using Context:

**Context** is data you pass in when you call `invoke`, such as a user's name. Think of it as fixed information for one run. The code can read it, but the run does not change it.

In the example below, a middleware function reads `user_name` from the context and builds the system prompt from it. The agent then addresses the user by name. The model sees the prompt text, not the context object itself.


*This data does not change during execution.*

[Reference](https://docs.langchain.com/oss/python/concepts/context#static-runtime-context)

In [53]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_openrouter import ChatOpenRouter


@dataclass
class ContextSchema:
    user_name: str

@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:  
    user_name = request.runtime.context.user_name
    
    return f"You are a helpful assistant. Address the user as {user_name}."


model = ChatOpenRouter(model="openai/gpt-6-luna")

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[personalized_prompt],
    context_schema=ContextSchema
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather like in Wexford?"}]},
    context=ContextSchema(user_name="Mo Che")  
)

In [54]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

what is the weather like in Wexford?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_ZHg9MQb0JNphRBxAhF2p6g2H)
 Call ID: call_ZHg9MQb0JNphRBxAhF2p6g2H
  Args:
    location: Wexford
================================= Tool Message =================================
Name: get_weather

Current weather in Wexford: 22 degrees C
================================== Ai Message ==================================

Mo Che, it’s currently 22°C in Wexford.


#### Using State:

**State** is the agent's working memory during a single run. It holds the list of messages and any extra fields you declare. In LangGraph, state acts as short-term memory during a run. Unlike context, state can be updated while the run is going, for example by tools or by the model's replies.

The example below only *reads* `user_name` from state. The value is passed in once at the start, so nothing changes during the run.


[Reference](https://docs.langchain.com/oss/python/concepts/context#dynamic-runtime-context)

In [55]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.agents import AgentState
from langchain_openrouter import ChatOpenRouter


class CustomState(AgentState):  
    user_name: str

@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:  
    user_name = request.state.get("user_name", "User")
    return f"You are a helpful assistant. User's name is {user_name}"

model = ChatOpenRouter(model="openai/gpt-6-luna")

agent = create_agent(
    model=model,
    state_schema=CustomState,  
    middleware=[personalized_prompt],  
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "hi!"}],
    "user_name": "Mo Che"
})

print(result)

{'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='fb7a35d2-bda3-4fff-a3c7-7bb7ee3b2b4d'), AIMessage(content='Hi, Mo! How can I help?', additional_kwargs={'reasoning_details': [{'data': 'gAAAAABq4gE_LgaMVHD-Fe-9-NSjIno2cXBKleyCbeUGOiocMYfRGoYGmZyCEas5pGYCbiIWCI7JQAmqkuV8iRWynmsOm5Uho-u-JzBWHvawul82Ih-p4j1O5ulUm0gxp53asTmwRMl36whicRKg8TT2XQYW5pa6FkwpcM5S9_IGu2NfVvj5k4-4T0fjUh3gKK4xVe-brKOALh0tSK6050479RgSvb7EH4UVk9jXmFjVuP2wwcHrX72vwQVuM1RTC_sXkTzXWzvkQdSndUrVURmnudtrJQJodkjkwIBg8_mtmzTbTQ_8_668iIiz2CdCz3rfn0TPvCahDcwl6nvm7-wBjzbIqdvtW9tE4Hf7ip7xGttq35uwFCLA3Nv9B_CgtNOhcI1o-Nxrdcg4Erk8zPbn8IsYo_xpvXgDg9_ROi_SC2nkMwrQlcWlR32TzD4H4c7nFOpvkRPjj0GzGE2FioxsssR8l02Aip6BafjK6ogpPnis6kfCKG6iMKvtANU5n-MobwSFZf596D4oHHvCK-ir-UhNxRC3SYDOoDN8xaBqsuXTfEpqRr3A3M5n4TXV2neoD45un-OvgZHgNw7xE8G9QRrYaPTjo38oS7FF-_n_ccF9HtQSMxH3jMe6R2dP5A2qGZCQg0IlPk-bKHHxwDURa2NQWCynhNKAUqKkVpxgIEJhyHnD0VsWpySlo7xeqs6l28_JOyNzKNhpGNVLwCCMkbyFO2Qd-WCtOyiMKqBiKbb7OQXEIkwPPaTh-Oa-RPHp5x

> Turning on memory: Memory is a powerful feature that allows you to persist the agent’s state across multiple invocations. Otherwise, the state is scoped only to a single run. Memory is covered further down in this notebook.

### [Dynamic system prompt](https://docs.langchain.com/oss/python/langchain/agents#dynamic-system-prompt)

The two examples above use `@dynamic_prompt`. It builds the system prompt fresh for each model call, using context or state, so the instructions can change from user to user.

*See also [Tool Strategy][Tool Strategy]*

[Tool Strategy]: https://docs.langchain.com/oss/python/langchain/agents#toolstrategy

### [Life-cycle Context](https://docs.langchain.com/oss/python/langchain/context-engineering#life-cycle-context)

Life-cycle hooks are functions that run at fixed points in an agent's run. `wrap_model_call` in the dynamic model example is one of them: it runs around each model call. The exercise below uses the same hook.

# Exercise

Create an agent that uses a different LLM model based on whether the user has a paid subscription or not.

**Hints:**

- Start from the `DynamicModelMiddleware` example. It swaps the model inside `wrap_model_call`.
- Read the subscription with `request.runtime.context.subscription`, as in the Using Context example.
- Pass the subscription in with `context=` when you call `invoke`.
- The skeleton names the class `Context`, but the examples above use `ContextSchema`. You can rename it, as long as you use the same name everywhere.

Test your agent with both a free and a paid subscriber. How could you confirm which model answered? (Hint: think about where you could add a `print` statement.)

In [ ]:

from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langgraph.runtime import Runtime

@dataclass
class Context:  
    subscription: str = "free"  # or "paid"

# use openai/gpt-6-luna for paid users, openai/gpt-5-nano for free users




# 🔄 **Agents and Memory**

### **Stateless Agent Characteristics**

- By default, each `invoke` call is independent. The agent does not remember earlier questions.

### **Enabling Memory**

- **Checkpointer**: a component that saves the agent's state so it can be reloaded later. We use `InMemorySaver`, which keeps the data in the computer's memory. The saved data is lost when the notebook restarts.

- **Thread ID**: a name for one conversation. Calls that use the same `thread_id` share the same saved state.

You don't need to name a history variable yourself. The checkpointer and the `thread_id` handle this.

🔑 **Takeaway**: To add memory, pass a checkpointer when you create the agent, and pass the same `thread_id` in `config` on every call in one conversation.

In [ ]:
# type: ignore

from langchain.agents import create_agent
from langchain_openrouter import ChatOpenRouter
from langgraph.checkpoint.memory import InMemorySaver  


model = ChatOpenRouter(model="openai/gpt-6-luna")

agent = create_agent(
    model=model,
    tools=[get_weather],
    checkpointer=InMemorySaver()
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather like in Wexford?"}]},
    config={"configurable": {"thread_id": "1"}}
)

result

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "in celsius please"}]},
    config={"configurable": {"thread_id": "1"}} # try without thread_id
)
result

#### Config

The `thread_id` in `config` tells the agent which conversation a call belongs to. `InMemorySaver` uses it to find the saved history for that conversation.

**Try it:** the second call above passes `thread_id` "1", and its comment suggests trying it without the `thread_id`. Remove the `config` argument, run the cell, and note what happens. Then put the `config` back.

**With the same `thread_id`:** the agent should be able to use earlier messages. "In celsius please" only makes sense because the first question named Wexford, so the agent needs that earlier message to answer well.